# OrganChip Insight — runtime probe only
Enable a GPU, disable Internet, attach **no datasets or weights**. Run all cells.
This notebook reports versions and checks MobileNetV3/CUDA and ONNX parity using random weights and synthetic tensors. It does not train or access data.
Download the generated `runtime-report.json`. A failed check is diagnostic; it does not alter dependencies or relax the training runtime contract.
Passing these checks is not scientific validation or a complete training preflight.
Version 2 additionally verifies and extracts the attached ONNX Runtime wheel without pip, network access, or mutation of the Kaggle base environment.


In [ ]:
"""Offline environment diagnostics with synthetic inputs and no trained weights."""

from __future__ import annotations

import argparse
import hashlib
import importlib.metadata
import json
import platform
import stat
import sys
import tempfile
import zipfile
from datetime import UTC, datetime
from pathlib import Path
from typing import Any


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def bootstrap_offline_wheel(wheel: Path, *, expected_sha256: str, target: Path) -> dict[str, Any]:
    if sha256_file(wheel) != expected_sha256:
        raise RuntimeError("Offline wheel checksum mismatch")
    if target.exists():
        raise RuntimeError("Offline wheel target already exists")
    target.mkdir(parents=True)
    with zipfile.ZipFile(wheel) as archive:
        members = archive.infolist()
        for member in members:
            path = Path(member.filename)
            mode = member.external_attr >> 16
            if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(mode) or not path.parts:
                raise RuntimeError(f"Unsafe offline wheel member: {member.filename}")
        archive.extractall(target)
    sys.path.insert(0, str(target))
    importlib.invalidate_caches()
    return {
        "wheel": wheel.name,
        "sha256": expected_sha256,
        "target": str(target),
        "member_count": len(members),
    }


def inspect_versions(contract: dict[str, Any]) -> dict[str, Any]:
    runtime = contract["runtime"]
    versions: dict[str, str | None] = {}
    blockers: list[str] = []
    for name in runtime["packages"]:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
            blockers.append(f"Missing package: {name}")
    try:
        from packaging.specifiers import SpecifierSet
        from packaging.version import Version

        python_spec = runtime["python"]["specifier"]
        if Version(platform.python_version()) not in SpecifierSet(python_spec):
            blockers.append(f"Python {platform.python_version()} violates {python_spec}")
        for name, version in versions.items():
            spec = runtime["packages"][name]["specifier"]
            if version is not None and Version(version) not in SpecifierSet(spec):
                blockers.append(f"{name} {version} violates {spec}")
        pair_ok = any(
            all(
                versions[name] is not None
                and Version(versions[name]) in SpecifierSet(f"=={pair[name]}")
                for name in ("torch", "torchvision")
            )
            for pair in runtime["accepted_torch_torchvision_pairs"]
        )
        if not pair_ok:
            blockers.append("Torch/torchvision pair is not accepted")
    except Exception as error:
        blockers.append(f"Version validation failed: {type(error).__name__}: {error}")
    return {"python": platform.python_version(), "packages": versions, "blockers": blockers}


def probe_runtime(
    contract: dict[str, Any],
    output_directory: Path,
    *,
    offline_wheel: Path | None = None,
    offline_wheel_sha256: str | None = None,
) -> dict[str, Any]:
    output_directory.mkdir(parents=True, exist_ok=True)
    run_directory = Path(tempfile.mkdtemp(prefix="organchip-runtime-probe-", dir=output_directory))
    report: dict[str, Any] = {
        "schema_version": 1,
        "scope": "runtime-only; synthetic inputs; random weights; no dataset or training",
        "benchmark_eligible": False,
        "generated_at_utc": datetime.now(UTC).isoformat(),
        "contract_id": contract["contract_id"],
        "contract_semantic_sha256": hashlib.sha256(
            json.dumps(contract, sort_keys=True, separators=(",", ":")).encode()
        ).hexdigest(),
        "platform": platform.system(),
        "versions": inspect_versions(contract),
        "checks": {},
    }

    if (offline_wheel is None) != (offline_wheel_sha256 is None):
        raise ValueError("Offline wheel path and SHA-256 must be supplied together")
    if offline_wheel is not None and offline_wheel_sha256 is not None:
        report["offline_bootstrap"] = bootstrap_offline_wheel(
            offline_wheel,
            expected_sha256=offline_wheel_sha256,
            target=run_directory / "offline-site-packages",
        )

    def check(name, operation):
        try:
            report["checks"][name] = {"passed": True, "details": operation()}
        except Exception as error:
            report["checks"][name] = {
                "passed": False,
                "error": f"{type(error).__name__}: {error}",
            }

    def cuda_forward():
        import torch
        from torchvision.models import mobilenet_v3_small

        if not torch.cuda.is_available():
            raise RuntimeError("CUDA unavailable: GPU validation remains blocked")
        torch.manual_seed(20260916)
        model = mobilenet_v3_small(weights=None, num_classes=2).eval().cuda()
        with torch.inference_mode():
            output = model(torch.randn(1, 3, 224, 224, device="cuda"))
        if list(output.shape) != [1, 2] or not torch.isfinite(output).all().item():
            raise RuntimeError("Invalid CUDA output")
        torch.cuda.synchronize()
        return {
            "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            "torch_cuda": torch.version.cuda,
            "cudnn": torch.backends.cudnn.version(),
            "output_shape": list(output.shape),
        }

    def onnx_round_trip():
        import numpy as np
        import onnx
        import onnxruntime as ort
        import torch
        from torchvision.models import mobilenet_v3_small

        torch.manual_seed(20260916)
        model = mobilenet_v3_small(weights=None, num_classes=2).eval().cpu()
        sample = torch.randn(2, 3, 224, 224)
        model_path = run_directory / "synthetic-model.onnx"
        torch.onnx.export(
            model,
            sample,
            model_path,
            input_names=["images"],
            output_names=["logits"],
            dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
            opset_version=18,
            do_constant_folding=True,
            dynamo=False,
        )
        onnx.checker.check_model(onnx.load(model_path))
        session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        errors = []
        for batch in (1, 2, 3):
            values = torch.randn(batch, 3, 224, 224)
            with torch.inference_mode():
                expected = model(values).numpy()
            actual = session.run(["logits"], {"images": values.numpy()})[0]
            if not np.isfinite(actual).all() or not np.isfinite(expected).all():
                raise RuntimeError("Non-finite parity output")
            np.testing.assert_allclose(actual, expected, rtol=0, atol=1e-4)
            errors.append(float(np.max(np.abs(actual - expected))))
        return {"batch_sizes": [1, 2, 3], "max_absolute_error": max(errors), "tolerance": 1e-4}

    # Diagnostic checks run even when version bounds fail; they never relax the contract.
    check("cuda_mobilenet_forward", cuda_forward)
    check("cpu_onnx_round_trip", onnx_round_trip)
    report["runtime_checks_passed"] = not report["versions"]["blockers"] and all(
        result["passed"] for result in report["checks"].values()
    )
    report_path = run_directory / "runtime-report.json"
    report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps(report, indent=2, sort_keys=True))
    print(f"Report: {report_path}")
    return report


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--contract", type=Path, required=True)
    parser.add_argument("--output-directory", type=Path, required=True)
    parser.add_argument("--offline-wheel", type=Path)
    parser.add_argument("--offline-wheel-sha256")
    args = parser.parse_args()
    probe_runtime(
        json.loads(args.contract.read_text(encoding="utf-8")),
        args.output_directory,
        offline_wheel=args.offline_wheel,
        offline_wheel_sha256=args.offline_wheel_sha256,
    )



In [ ]:
CONTRACT = json.loads('{\n  "schema_version": 1,\n  "contract_id": "organchip-ooc-cnn-kaggle-runtime-v1",\n  "execution_policy": {\n    "internet_required": false,\n    "network_installation_allowed": false,\n    "implicit_model_downloads_allowed": false,\n    "test_access_receipt_scope": "workspace",\n    "test_manifest_access_modes": [\n      "final-eval"\n    ]\n  },\n  "modes": {\n    "smoke": {\n      "allowed_devices": [\n        "cpu",\n        "cuda"\n      ],\n      "gpu_required": false,\n      "pretrained_weights_required": false\n    },\n    "validation": {\n      "allowed_devices": [\n        "cuda"\n      ],\n      "gpu_required": true,\n      "pretrained_weights_required": true\n    },\n    "final-eval": {\n      "allowed_devices": [\n        "cuda"\n      ],\n      "gpu_required": true,\n      "pretrained_weights_required": false,\n      "frozen_manifest_required": true,\n      "test_access_receipt_required": true,\n      "test_access_receipt_scope": "workspace"\n    }\n  },\n  "runtime": {\n    "python": {\n      "specifier": ">=3.11,<3.13",\n      "locally_validated_version": "3.12.14"\n    },\n    "packages": {\n      "torch": {\n        "specifier": ">=2.6,<2.11",\n        "locally_validated_version": "2.10.0"\n      },\n      "torchvision": {\n        "specifier": ">=0.21,<0.26",\n        "locally_validated_version": "0.25.0"\n      },\n      "numpy": {\n        "specifier": ">=2.0,<2.4",\n        "locally_validated_version": "2.3.5"\n      },\n      "pandas": {\n        "specifier": ">=2.2,<3.1",\n        "locally_validated_version": "3.0.5"\n      },\n      "Pillow": {\n        "specifier": ">=11,<13",\n        "locally_validated_version": "12.3.0"\n      },\n      "scikit-learn": {\n        "specifier": ">=1.6,<2",\n        "locally_validated_version": "1.9.1"\n      },\n      "matplotlib": {\n        "specifier": ">=3.9,<3.12",\n        "locally_validated_version": "3.11.1"\n      },\n      "onnx": {\n        "specifier": ">=1.17,<1.23",\n        "locally_validated_version": "1.22.0"\n      },\n      "onnxruntime": {\n        "specifier": ">=1.20,<1.23",\n        "locally_validated_version": "1.22.2"\n      },\n      "packaging": {\n        "specifier": ">=24,<27",\n        "locally_validated_version": "26.3"\n      }\n    },\n    "tooling_packages": {\n      "pytest": {\n        "specifier": ">=8.4,<9",\n        "cpu_environment_version": "8.4.2"\n      },\n      "ruff": {\n        "specifier": ">=0.16,<0.17",\n        "cpu_environment_version": "0.16.7"\n      },\n      "nbformat": {\n        "specifier": ">=5.10,<6",\n        "cpu_environment_version": "5.10.4"\n      },\n      "nbclient": {\n        "specifier": ">=0.10,<0.11",\n        "cpu_environment_version": "0.10.2"\n      },\n      "ipykernel": {\n        "specifier": ">=6.31,<7",\n        "cpu_environment_version": "6.31.0"\n      }\n    },\n    "accepted_torch_torchvision_pairs": [\n      {\n        "torch": "2.6.*",\n        "torchvision": "0.21.*"\n      },\n      {\n        "torch": "2.7.*",\n        "torchvision": "0.22.*"\n      },\n      {\n        "torch": "2.8.*",\n        "torchvision": "0.23.*"\n      },\n      {\n        "torch": "2.9.*",\n        "torchvision": "0.24.*"\n      },\n      {\n        "torch": "2.10.*",\n        "torchvision": "0.25.*"\n      }\n    ]\n  },\n  "local_inputs": {\n    "source_bundle_required": true,\n    "dataset_required": true,\n    "split_lock_required": true,\n    "initial_weights": {\n      "required_for_modes": [\n        "validation"\n      ],\n      "source": "explicit-local-file",\n      "torchvision_default_weights_allowed": false,\n      "required_metadata": [\n        "path",\n        "sha256",\n        "architecture",\n        "weight_enum",\n        "source_url",\n        "license"\n      ]\n    }\n  },\n  "required_hashes": {\n    "all_modes": [\n      "source_bundle_sha256",\n      "runtime_contract_sha256",\n      "config_sha256",\n      "dataset_inventory_sha256",\n      "split_lock_sha256",\n      "train_validation_manifest_sha256"\n    ],\n    "validation": [\n      "initial_weights_sha256"\n    ],\n    "final-eval": [\n      "frozen_manifest_sha256",\n      "checkpoint_sha256",\n      "test_manifest_sha256"\n    ]\n  },\n  "preflight": {\n    "fail_if_gpu_missing_in_modes": [\n      "validation",\n      "final-eval"\n    ],\n    "fail_if_network_installation_requested": true,\n    "fail_if_implicit_weight_download_requested": true,\n    "fail_if_required_hash_missing": true,\n    "fail_if_hash_mismatch": true,\n    "fail_if_torchvision_pair_unlisted": true,\n    "capture": [\n      "python_version",\n      "package_versions",\n      "pip_freeze",\n      "cuda_version",\n      "cudnn_version",\n      "gpu_name",\n      "gpu_count"\n    ]\n  }\n}\n')
OFFLINE_WHEEL = Path('/kaggle/input/datasets/oumarbenlol/organchip-cnn-offline-resources-v1/resources/onnxruntime-1.22.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl')
OFFLINE_WHEEL_SHA256 = '2d39a530aff1ec8d02e365f35e503193991417788641b184f5b1e8c9a6d5ce8d'
OUTPUT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
runtime_report = probe_runtime(
    CONTRACT, OUTPUT_ROOT, offline_wheel=OFFLINE_WHEEL,
    offline_wheel_sha256=OFFLINE_WHEEL_SHA256,
)
